# Task 1: Titanic Passenger Survival Prediction
## 1. Introduction and Data Loading
In this section, we load the historical Titanic dataset. The goal is to build a binary classification model that predicts whether a passenger survived (1) or did not survive (0) based on features like age, ticket class, and gender.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Loading the data
df = pd.read_csv('train.csv')

# First few rows to understand the structure
print(df.head())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


## 2. Exploratory Data Analysis & Cleaning
Real-world data is rarely perfectly clean. Before modeling, we must handle missing values and understand the distribution of our features. We will impute missing ages and drop columns that lack predictive power or have too many missing values (like Cabin numbers).

In [2]:
# Checking the missing data
print("Missing values before cleaning:\n", df.isnull().sum())

# Age is missing a lot. Filling with the median age so outliers don't skew it.
df['Age'] = df['Age'].fillna(df['Age'].median())

# Cabin has too many missing values to be useful, and Ticket is just random strings.
df = df.drop(columns=['Cabin', 'Ticket'])

# Dropping few rows where 'Embarked' is missing
df = df.dropna(subset=['Embarked'])

print("\nMissing values after cleaning:\n", df.isnull().sum())

Missing values before cleaning:
 PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Missing values after cleaning:
 PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Fare           0
Embarked       0
dtype: int64


## 3. Feature Engineering
Machine learning models require numerical input. We need to convert categorical text data (like 'male'/'female') into numerical formats using encoding techniques.

In [3]:
# Mapping Sex to 0 and 1.
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

# drop_first=True prevents the dummy variable trap
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)

# Selecting features used for training
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked_Q', 'Embarked_S']
X = df[features]
y = df['Survived']

## 4. Model Building and Training
For this classification task, we will utilize a standard machine learning algorithm. We split our data into a training set to teach the model, and a validation set to test its unseen accuracy.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Split data: 80% for training, 20% for testing our model's accuracy
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialization of model.
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Training the model
model.fit(X_train, y_train)
print("Model training complete.")

Model training complete.


## 5. Evaluation
Accuracy alone isn't always enough. We evaluate the model using a classification report to understand its precision and recall—meaning how well it correctly identified both survivors and non-survivors.

In [5]:
from sklearn.metrics import accuracy_score, classification_report

# Predictions on our test set
predictions = model.predict(X_test)

# Results
print("Accuracy:", accuracy_score(y_test, predictions))
print("\nClassification Report:\n", classification_report(y_test, predictions))

Accuracy: 0.7696629213483146

Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.78      0.81       109
           1       0.68      0.75      0.72        69

    accuracy                           0.77       178
   macro avg       0.76      0.77      0.76       178
weighted avg       0.78      0.77      0.77       178

